Read >= 1 million rows of data, >= 50 columns, with time processing <= 0.5s

In [ ]:
import pandas as pd
import numpy as np
from loguru import logger
import time

In [ ]:
# 1. change format from .xlsx to parquet or feather
# 2. use chunksize
# 3. use dask
# 4. use polars

In [ ]:
def xlsx_to_parquet(file_path: str = "", sheet_name: str = ""):
    df = pd.read_excel(file_path, sheet_name=sheet_name)
    df.to_parquet(file_path + ".parquet")

def xlsx_to_feather(file_path: str = "", sheet_name: str = ""):
    df = pd.read_excel(file_path, sheet_name=sheet_name)
    df.to_feather(file_path + ".feather")

In [ ]:
import pandas as pd
import numpy as np
from loguru import logger
import time
from functools import wraps
import sys

logger.remove()
logger.add(sys.stdout, level="DEBUG")
logger.info("Hello from loguru")

def time_counter(running_time: int = 100):
    """
    A decorator that measures the average execution time of a function over a specified number of runs.

    Args:
        running_time (int): The number of times to run the function.
    """
    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            if running_time <= 0:
                return func(*args, **kwargs)

            total_time: float = 0
            result = None

            for i in range(running_time):
                start = time.perf_counter()
                result = func(*args, **kwargs)
                end = time.perf_counter()
                total_time += end - start

            avg_time = total_time / running_time

            logger.info(f"Function '{func.__name__}' was run {running_time} times.")
            logger.info(f"Total execution time: {total_time} seconds.")
            logger.info(f"Average execution time: {avg_time} seconds per run.")

            return result
        return wrapper
    return decorator

In [ ]:
@time_counter(running_time=100)
def test():
    return 1
test()

In [ ]:
# create a dataframe with 1 million rows and 50 columns
file_name = "test_read_data.xlsx"
df = pd.DataFrame(np.zeros((1_000_000, 50), dtype=np.int16))
df.to_excel(file_name, index=False)

In [ ]:
# read by normal pandas.read_excel()
@time_counter(10)
def read_excel_normal(file_path: str = "", sheet_name: str = ""):
    df = pd.read_excel(file_path, sheet_name=sheet_name)
    return df

read_excel_normal(file_name)

In [ ]:
# read by chunksize
@time_counter(10)
def read_excel_chunksize(file_path: str = "", sheet_name: str = "", chunksize: int = 100_000):
    df = pd.DataFrame()
    for chunk in pd.read_excel(file_path, sheet_name=sheet_name, chunksize=chunksize):
        df = pd.concat([df, chunk])
    return df

read_excel_chunksize(file_name)

In [ ]:
# read by dask
import dask.dataframe as dd

@time_counter(10)
def read_excel_dask(file_path: str = "", sheet_name: str = ""):
    df = dd.read_excel(file_path, sheet_name=sheet_name)
    df = df.compute()
    return df

read_excel_dask(file_name)

In [ ]:
# read by polars
import polars as pl

@time_counter(10)
def read_excel_polars(file_path: str = "", sheet_name: str = ""):
    df = pl.read_excel(file_path, sheet_name=sheet_name)
    df = df.to_pandas()
    return df

read_excel_polars(file_name)

In [ ]:
# read by parquet
xlsx_to_parquet(file_name)

@time_counter(10)
def read_excel_parquet(file_path: str = "", sheet_name: str = ""):
    df = pd.read_parquet(file_path, sheet_name=sheet_name)
    return df

read_excel_parquet(file_name + ".parquet")

In [ ]:
xlsx_to_feather(file_name)

# read by feather
@time_counter(100)
def read_excel_feather(file_path: str = "", sheet_name: str = ""):
    df = pd.read_feather(file_path, sheet_name=sheet_name)
    return df

read_excel_feather(file_name + ".feather")

In [ ]:


# read by pickle
@time_counter(100)
def read_excel_pickle(file_path: str = "", sheet_name: str = ""):
    df = pd.read_pickle(file_path, sheet_name=sheet_name)
    return df
